# 01. vLLM 기초 이해

이 노트북에서는 **vLLM이 무엇인지**, **왜 빠른지**, **어떻게 사용하는지**를 단계적으로 배웁니다.

---

## ⚙️  M2 Pro (Apple Silicon) 설치 방법

**vLLM 0.22.0+cpu** 는 macOS Apple Silicon을 **공식 지원**하는 CPU 전용 빌드입니다.  
C++ 컴파일 없이 설치 가능합니다 (~3분).

```bash
bash scripts/install_vllm_local.sh
```

> **0.22.0+cpu 란?**  
> vLLM 팀이 CPU 환경(Linux / macOS ARM)을 위해 별도로 배포하는 빌드입니다.  
> `platforms/__init__.py` 안에 macOS 자동 감지 코드가 포함되어 있어  
> 별도 설정 없이 M2 Pro에서 바로 실행됩니다.  
> Cloudera CML 브랜치(GPU)에는 일반 `vllm` 빌드를 사용합니다.

---

## vLLM이란?

vLLM(Virtual Large Language Model)은 LLM을 **빠르고 효율적으로 서빙**하기 위한 오픈소스 엔진입니다.  
UC Berkeley에서 개발했으며, **OpenAI API와 완전히 호환**됩니다.

---

## 일반 추론 vs vLLM 비교

| 항목 | 일반 추론 (Hugging Face) | vLLM |
|------|-------------------------|------|
| 메모리 관리 | 고정 블록 → 낭비 발생 | **PagedAttention** → 낭비 없음 |
| 요청 처리 | 배치 단위 처리 (기다림) | **Continuous Batching** → 즉시 처리 |
| API | 직접 구현 필요 | **OpenAI 호환 API** 내장 |
| 처리량 (Throughput) | 기준 1x | **최대 24x 빠름** |

---

## 핵심 개념: PagedAttention

```
일반 방식:
┌─────────────────────────────┐
│  요청 A의 KV Cache (고정)    │  ← 빈 공간도 미리 예약 → 낭비
│  [토큰1][토큰2][   ][   ]    │
└─────────────────────────────┘

PagedAttention:
┌──────┐ ┌──────┐ ┌──────┐
│ 블록1 │ │ 블록2 │ │ 블록3 │  ← 필요할 때만 페이지 할당
│A토큰  │ │A토큰  │ │B토큰  │  ← 다른 요청이 같은 블록 공유 가능
└──────┘ └──────┘ └──────┘
```

→ GPU 메모리를 **훨씬 효율적으로** 사용하므로 더 많은 동시 요청 처리 가능

---

## 핵심 개념: Continuous Batching

```
일반 배치:
시간 →  [요청A 완료 대기]---[요청B 시작]
         배치가 끝날 때까지 새 요청 못 받음

Continuous Batching:
시간 →  [요청A 처리 중][요청B 중간에 합류][요청C 합류]
         토큰 생성 도중에도 새 요청을 즉시 받아서 처리
```

→ 서버 활용률이 높아져 **전체 처리량(throughput)** 이 크게 향상됨

## Cell 1. vLLM 서버 시작 명령어 이해

아래는 vLLM 서버를 시작하는 명령어입니다.  
각 파라미터가 무엇을 의미하는지 주석으로 설명합니다.

```bash
python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen2.5-3B-Instruct \   # 사용할 모델 (HuggingFace Hub 경로)
  --device cpu \                         # 실행 장치: cpu / cuda / mps
  --dtype float32 \                      # 숫자 정밀도: float32(CPU) / bfloat16(GPU)
  --max-model-len 4096 \                 # 처리할 수 있는 최대 토큰 길이
  --host 0.0.0.0 \                       # 외부에서 접속 허용
  --port 8000                            # API 서버 포트
```

**각 파라미터 의미:**
- `--model`: 어떤 모델을 쓸지. HuggingFace Hub에서 자동 다운로드됩니다.
- `--device cpu`: M2 Pro 로컬 환경에서는 CPU 모드 사용 (GPU가 없으므로)
- `--dtype float32`: CPU는 float32가 안정적. GPU는 bfloat16으로 속도 향상 가능
- `--max-model-len 4096`: 한 번에 처리할 수 있는 최대 토큰 수. 클수록 메모리 필요

In [ ]:
# vLLM 서버가 실행 중인지 확인합니다.
# 이 셀을 실행하기 전에 scripts/start_vllm.sh 를 먼저 실행하세요.

import requests

VLLM_BASE_URL = "http://localhost:8000"

try:
    response = requests.get(f"{VLLM_BASE_URL}/health", timeout=5)
    print(f"✅ vLLM 서버 상태: {response.status_code}")
    print("서버가 정상적으로 실행 중입니다.")
except requests.ConnectionError:
    print("❌ vLLM 서버에 연결할 수 없습니다.")
    print("→ 터미널에서 먼저 실행하세요: bash scripts/start_vllm.sh")

## Cell 2. 로드된 모델 확인

vLLM 서버가 현재 어떤 모델을 로드했는지 확인합니다.  
`GET /v1/models` 엔드포인트는 OpenAI API와 동일한 형식으로 응답합니다.

In [ ]:
import requests
import json

VLLM_BASE_URL = "http://localhost:8000"

# GET /v1/models → 현재 로드된 모델 목록 조회
response = requests.get(f"{VLLM_BASE_URL}/v1/models")
models = response.json()

print("=== 현재 로드된 모델 ===")
for model in models["data"]:
    print(f"  모델 ID : {model['id']}")
    print(f"  오브젝트 : {model['object']}")
    print(f"  생성 시각 : {model['created']}")

## Cell 3. 첫 번째 API 호출 — Hello vLLM!

이제 실제로 vLLM에 질문을 보내봅니다.  
**OpenAI SDK**를 그대로 사용하되, `base_url`만 vLLM 서버로 바꾸면 됩니다.

```
내 코드  →  openai SDK  →  vLLM 서버 (localhost:8000)  →  모델 추론  →  응답
                            (OpenAI API와 동일한 형식)
```

In [ ]:
from openai import OpenAI

# OpenAI SDK를 vLLM 서버에 연결
# base_url만 바꾸면 OpenAI → vLLM으로 전환 완료
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy",  # vLLM은 API 키가 필요 없지만, SDK 형식을 맞추기 위해 넣음
)

# 모델 이름은 /v1/models 에서 확인한 ID와 동일해야 합니다
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# 첫 번째 요청
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "안녕하세요! 자기소개를 한 문장으로 해주세요."}
    ],
    max_tokens=100,
)

print("=== AI 응답 ===")
print(response.choices[0].message.content)

## Cell 4. 응답 구조 분석

vLLM(OpenAI 호환 API)의 응답 객체가 어떻게 생겼는지 살펴봅니다.  
특히 **토큰 사용량(usage)** 은 비용 계산과 성능 최적화에 중요합니다.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy",
)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "파이썬이란 무엇인가요? 두 문장으로 설명해주세요."}
    ],
    max_tokens=150,
)

print("=" * 50)
print("📋 응답 구조 분석")
print("=" * 50)

# 응답 텍스트
print(f"\n💬 응답 내용:")
print(f"  {response.choices[0].message.content}")

# 응답이 어떻게 끝났는지 (stop, length, content_filter)
print(f"\n🛑 종료 이유 (finish_reason):")
print(f"  {response.choices[0].finish_reason}")
print(f"  → 'stop' : 문장이 자연스럽게 완성됨")
print(f"  → 'length': max_tokens에 도달해서 잘림")

# 토큰 사용량
print(f"\n🔢 토큰 사용량 (usage):")
print(f"  입력 토큰 (prompt_tokens)     : {response.usage.prompt_tokens}")
print(f"  출력 토큰 (completion_tokens) : {response.usage.completion_tokens}")
print(f"  합계 (total_tokens)           : {response.usage.total_tokens}")
print(f"\n  ※ 토큰 = 단어/글자의 기본 단위. 한국어 1글자 ≈ 1~2토큰")

## Cell 5. 시스템 프롬프트로 역할 부여

**시스템 프롬프트(system prompt)** 는 AI의 역할과 말투를 설정합니다.  
이 프로젝트의 핵심 기능인 "패션 스타일리스트 챗봇"이 여기서 시작됩니다.

```
messages 배열 구조:
[
  {"role": "system",    "content": "AI의 역할 정의"},   ← 시스템 프롬프트
  {"role": "user",      "content": "사용자 질문"},       ← 유저 메시지
  {"role": "assistant", "content": "AI 이전 답변"},      ← 대화 히스토리 (선택)
  {"role": "user",      "content": "사용자 다음 질문"},   ← 현재 질문
]
```

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy",
)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# 시스템 프롬프트로 AI에게 역할 부여
SYSTEM_PROMPT = """
너는 10년 경력의 청담동 패션 스타일리스트야.
친근한 말투로 트렌디한 브랜드를 섞어서 추천해 줘.
답변은 3~5문장으로 간결하게 해줘.
"""

# 시스템 프롬프트 없을 때 vs 있을 때 비교
user_question = "내일 격식 있는 자리에 입고 갈 만한 30대 남성 비즈니스 캐주얼 추천해줘"

print("=" * 50)
print("[시스템 프롬프트 없이]")
print("=" * 50)
response_no_system = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": user_question}],
    max_tokens=200,
)
print(response_no_system.choices[0].message.content)

print("\n" + "=" * 50)
print("[패션 스타일리스트 시스템 프롬프트 적용]")
print("=" * 50)
response_with_system = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_question},
    ],
    max_tokens=200,
)
print(response_with_system.choices[0].message.content)

## 정리

이 노트북에서 배운 것:

1. **vLLM의 핵심**: PagedAttention(메모리 효율) + Continuous Batching(처리량 향상)
2. **서버 기동**: `--model`, `--device`, `--dtype`, `--max-model-len` 파라미터 의미
3. **OpenAI SDK 연결**: `base_url`만 바꾸면 OpenAI → vLLM 전환
4. **응답 구조**: `choices[0].message.content`, `usage.total_tokens`, `finish_reason`
5. **시스템 프롬프트**: AI의 역할/말투를 `role: system`으로 정의

---

다음 노트북: **`02_inference_params.ipynb`** — 추론 파라미터 실험